# RL Group Project 3 — Shared setup (start every notebook from this file)

**What this is.** The code every part of the project shares: environments, seeds, frame preprocessing, the training log, and the 200-game evaluation. Pong Part 1 (`pong_full`), Pong Part 2 (`pong_5k`), Breakout (`breakout`) and the final report all use it, so every agent is scored the same way and the TA's re-run reproduces our numbers.

**How to use it.**
1. Copy this notebook and rename the copy for your part: `pong_full.ipynb`, `pong_5k.ipynb` or `breakout.ipynb`.
2. Leave the cells above the **"Your agent starts here"** line as they are. If something shared needs to change, change it here in the repo and tell the group, so all four notebooks stay identical.
3. Build your agent below that line.
4. At the end, D pastes this shared section **once** at the top of the final submission notebook, followed by each part's code. That gives the one `.ipynb` + HTML that Canvas asks for.

**What's in here**

| Section | What it gives you | Who uses it |
| --- | --- | --- |
| Constants | Folder names, 200 eval games, eval seeds, frameskip 4, no sticky actions | Everyone |
| `set_global_seeds(seed)` | Seeds Python, NumPy, PyTorch; deterministic GPU math | Everyone |
| `get_device()` | CUDA (Colab) → Apple MPS → CPU | Everyone |
| `make_env(game)` | `ALE/<game>-v5`, an action every 4 frames | Everyone |
| `preprocess_pong`, `PongFrameDiff`, `ball_visible_pong` | 80×80 Pong frames, frame differencing, "is the ball on screen?" | A, B |
| `preprocess_breakout`, `breakout_fire_needed` | 82×72 Breakout frames, auto-FIRE after reset / lost life | C |
| `FrameStack(preprocess_fn, k=4)` | Stacks the last k frames so the net sees motion | A, B, C |
| `TrainingLogger(agent, resume=True)` | `results/<agent>_train_log.csv` + `_meta.json`; resumes after a Colab disconnect | A, B, C |
| `fetch_weights(agent)` | Downloads `models/<agent>.pt` from the GitHub repo if it isn't on disk (so the TA can run with `TRAIN = False`) | A, B, C, D |
| `evaluate(env_fn, policy, agent_name=...)` | The 200-game greedy evaluation → `results/<agent>_eval_200.csv` | A, B, C |
| `random_baseline(game)` | Random-play baseline → `results/<game>_random_eval_200.csv` | C, D |

**Your policy object** only needs `policy.act(obs) -> int` (`obs` is the **raw** env frame; do your preprocessing inside) and, optionally, `policy.reset()` to clear frame-diff / frame-stack state between games. For evaluation, `act` must be greedy: no epsilon, no sampling.

**Files every agent delivers to D**

| File | Contents |
| --- | --- |
| `results/<agent>_train_log.csv` | One row per training game: game, score, 100-game running mean, frames, wall-clock seconds |
| `results/<agent>_eval_200.csv` | 200 rows: game, score, steps |
| `results/<agent>_meta.json` | Hyperparameters, training games, training hours, hardware, seed |
| `models/<agent>.pt` | Final trained weights (commit to the repo; keep each under ~50 MB) |
| `notes/<agent>_writeup.md` | 5–10 bullets on what you tried / what worked / what failed, plus 2–4 key code chunks |

Agent names: `pong_full`, `pong_5k`, `breakout`. Pong-5k also delivers `results/pong_5k_checkpoint_evals.csv`.

Checkpoints during training go in `models/checkpoints/`. That folder is git-ignored because checkpoints with replay buffers get large.

## Install
Uncomment on Colab or on a fresh machine. The versions match `requirements.txt` in the repo.

In [ ]:
# %pip install -q "gymnasium[atari]==1.4.0" "ale-py==0.12.1" "numpy==2.5.5" "pandas==3.0.1" "torch==2.12.0" "matplotlib==3.10.8"

In [ ]:
import json
import os
import platform
import random
import time
import urllib.request

import numpy as np
import pandas as pd
import gymnasium as gym
import ale_py
import torch

gym.register_envs(ale_py)  # registers the "ALE/..." environment ids with gymnasium

print(f"gymnasium {gym.__version__} | ale-py {ale_py.__version__} | numpy {np.__version__} "
      f"| pandas {pd.__version__} | torch {torch.__version__}")

## Constants
Every setting the whole team must agree on lives here, in one place.

In [ ]:
RESULTS_DIR = "results"            # CSV / JSON outputs
MODELS_DIR = "models"              # final weights: models/<agent>.pt
CHECKPOINT_DIR = os.path.join(MODELS_DIR, "checkpoints")  # mid-training saves (git-ignored)
NOTES_DIR = "notes"                # notes/<agent>_writeup.md

EVAL_GAMES = 200          # the assignment requires 200 evaluation games
EVAL_BASE_SEED = 10_000   # evaluation game i is reset with seed EVAL_BASE_SEED + i
FRAMESKIP = 4             # the agent picks an action every 4 frames; the env repeats it 4 times
STICKY_ACTIONS = 0.0      # 0 = deterministic emulator (ALE v5 defaults to 0.25 = random action repeats)

# Raw GitHub URL that fetch_weights() downloads from when a model file is missing locally.
WEIGHTS_BASE_URL = "https://raw.githubusercontent.com/sowmya-subra/rl-opt-project/main/models"

for d in (RESULTS_DIR, MODELS_DIR, CHECKPOINT_DIR, NOTES_DIR):
    os.makedirs(d, exist_ok=True)

## Seeding and device
`set_global_seeds` makes training runs repeatable and, more importantly, makes the greedy evaluation give **exactly** the same scores on every re-run. On a GPU, tiny floating-point differences can flip which action has the highest Q-value, so we also turn on PyTorch's deterministic mode.

In [ ]:
def set_global_seeds(seed):
    """Seed Python, NumPy and PyTorch, and ask PyTorch for deterministic GPU kernels."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # needed for deterministic cuBLAS
    torch.manual_seed(seed)                     # also seeds CUDA and MPS
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)  # warn instead of crash if an op has no deterministic version


def get_device():
    """Pick the fastest available device: CUDA (Colab GPU), then Apple Silicon (MPS), then CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def hardware_string():
    """Short description of the machine, recorded in <agent>_meta.json for the cost comparison."""
    cpu = platform.processor() or platform.machine()
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
    elif torch.backends.mps.is_available():
        gpu = "Apple MPS"
    else:
        gpu = "none"
    return f"{platform.system()} {cpu} | GPU: {gpu}"

## Environments
`frameskip=4` is how we "choose an action every 4 frames and play it 4 frames in a row". Sticky actions are off so a given seed always plays out the same way.

In [ ]:
def make_env(game, seed=None, frameskip=FRAMESKIP, sticky=STICKY_ACTIONS, render_mode=None):
    """
    Create an Atari env, e.g. make_env("Pong") or make_env("Breakout").

    frameskip   : frames each chosen action is repeated for (4 = act every 4 frames).
                  Use frameskip=1 to handle frame skipping yourself.
    sticky      : probability the emulator repeats the previous action (0 = off).
    seed        : if given, reset once with it so the env's RNG is fixed.
    render_mode : "rgb_array" if you want to record video, else None.
    """
    env = gym.make(f"ALE/{game}-v5", frameskip=frameskip,
                   repeat_action_probability=sticky, render_mode=render_mode)
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
    return env

## Preprocessing: Pong
Pong frames are 210×160 RGB. We crop off the scoreboard, halve the resolution, and turn everything into 0 (background) or 1 (paddles and ball). Pong actions: 0 NOOP, 1 FIRE, 2 RIGHT (= up), 3 LEFT (= down), 4 RIGHTFIRE, 5 LEFTFIRE.

In [ ]:
def preprocess_pong(frame):
    """
    210x160x3 RGB Pong frame -> 80x80 float32 array of 0/1.
    Crops the scoreboard and bottom border, downsamples by 2,
    erases the background, and sets paddles + ball to 1.
    """
    f = frame[35:195]            # crop: drop scoreboard (top) and border (bottom) -> 160x160
    f = f[::2, ::2, 0].copy()    # keep every 2nd pixel and only the red channel -> 80x80
    f[f == 144] = 0              # erase background (shade 1)
    f[f == 109] = 0              # erase background (shade 2)
    f[f != 0] = 1                # paddles and ball -> 1
    return f.astype(np.float32)


def ball_visible_pong(processed):
    """
    True if the ball is on screen. Use it to skip frames with no ball
    (e.g. right after a point), which carry nothing to learn from.
    Ignores the two paddle columns (8-9 = opponent, 70-71 = us). The rare frame
    where the ball sits exactly in a paddle column reads as "no ball".
    """
    return np.delete(processed, [8, 9, 70, 71], axis=1).any()


class PongFrameDiff:
    """
    Turns raw Pong frames into 'current minus previous' images, so one 80x80
    input shows which way the ball is moving (the approach from class).

        fd = PongFrameDiff()
        fd.reset()
        x = fd(obs)   # 80x80 float32, values in {-1, 0, 1}
    """
    def __init__(self):
        self.prev = None

    def reset(self):
        self.prev = None

    def __call__(self, frame):
        cur = preprocess_pong(frame)
        diff = cur - self.prev if self.prev is not None else np.zeros_like(cur)
        self.prev = cur
        return diff

## Preprocessing: Breakout, and frame stacking for any game
Breakout actions: 0 NOOP, 1 FIRE, 2 RIGHT, 3 LEFT. The ball must be launched with FIRE after a reset and after every lost life.

In [ ]:
def preprocess_breakout(frame):
    """
    210x160x3 RGB Breakout frame -> 82x72 float32 grayscale in [0, 1].
    Crops the score bar and side walls, downsamples by 2.
    """
    gray = frame.mean(axis=2)            # RGB -> grayscale
    f = gray[32:196, 8:152][::2, ::2]    # crop to the play area, keep every 2nd pixel
    return (f / 255.0).astype(np.float32)


class FrameStack:
    """
    Keeps the last k preprocessed frames, stacked as (k, H, W), so the
    network can see motion and direction. Works for any game.

        fs = FrameStack(preprocess_pong, k=4)
        x = fs.reset(first_obs)   # (4, 80, 80), first frame repeated 4 times
        x = fs(obs)               # (4, 80, 80), oldest frame dropped
    """
    def __init__(self, preprocess_fn, k=4):
        self.preprocess_fn = preprocess_fn
        self.k = k
        self.frames = []

    def reset(self, obs=None):
        self.frames = []
        if obs is not None:
            return self(obs)

    def __call__(self, obs):
        f = self.preprocess_fn(obs)
        if not self.frames:
            self.frames = [f] * self.k     # start of a game: pad with copies of the first frame
        else:
            self.frames = self.frames[1:] + [f]
        return np.stack(self.frames, axis=0)


def breakout_fire_needed(info, prev_lives):
    """
    True if Breakout just started or a life was just lost, i.e. the agent must
    press FIRE (action 1) to launch the ball. Inside your episode loop:

        if breakout_fire_needed(info, prev_lives):
            obs, *_ = env.step(1)
        prev_lives = info["lives"]
    """
    return prev_lives is None or info.get("lives", prev_lives) < prev_lives

## Training log
One row per training game. With `resume=True`, the logger reloads the existing CSV after a Colab disconnect, so the game count and wall-clock time pick up where they stopped. That matters for proving Pong-5k's 5,000-game cap and for reporting training cost.

In [ ]:
class TrainingLogger:
    """
    Records one row per training game and writes results/<agent>_train_log.csv.

        logger = TrainingLogger("pong_full", resume=True)
        logger.log(score, frames)          # once at the end of every game
        logger.games_played()              # e.g. to resume a training loop
        logger.save_meta(hyperparams, seed=SEED)

    resume=True  : continue an existing log (game numbers and wall-clock seconds carry on).
    resume=False : start a fresh log (overwrites the old CSV on the next save).
    The CSV is written every `save_every` games; call logger.save() right before saving
    a checkpoint so the log and the weights always match.
    """
    def __init__(self, agent_name, resume=True, save_every=10, window=100):
        self.agent = agent_name
        self.save_every = save_every
        self.window = window
        self.path = os.path.join(RESULTS_DIR, f"{agent_name}_train_log.csv")
        self.rows = []
        prior_seconds = 0.0
        if resume and os.path.exists(self.path):
            self.rows = pd.read_csv(self.path).to_dict("records")
            if self.rows:
                prior_seconds = float(self.rows[-1]["wall_seconds"])
            print(f"TrainingLogger: resuming {agent_name} at game {len(self.rows)}")
        # Shift the clock back so wall_seconds continues from the previous session.
        self.t0 = time.time() - prior_seconds

    def log(self, score, frames=None, **extra):
        """Record one finished game. Extra keyword args (e.g. epsilon=0.1) become extra columns."""
        game = len(self.rows) + 1
        recent = [r["score"] for r in self.rows[-(self.window - 1):]] + [score]
        row = {
            "game": game,
            "score": float(score),
            "running_mean_100": float(np.mean(recent)),
            "frames": frames,
            "wall_seconds": round(time.time() - self.t0, 1),
            **extra,
        }
        self.rows.append(row)
        if game % self.save_every == 0:
            self.save()
        return row

    def save(self):
        pd.DataFrame(self.rows).to_csv(self.path, index=False)

    def games_played(self):
        return len(self.rows)

    def truncate(self, n_games):
        """Drop rows after game n_games, e.g. when resuming from a checkpoint older than the log."""
        self.rows = self.rows[:n_games]
        if self.rows:
            self.t0 = time.time() - float(self.rows[-1]["wall_seconds"])
        self.save()

    def save_meta(self, hyperparams, seed=None, notes=""):
        """Write results/<agent>_meta.json with hyperparameters, training time and hardware."""
        meta = {
            "agent": self.agent,
            "training_games": len(self.rows),
            "training_hours": round((time.time() - self.t0) / 3600, 3),
            "seed": seed,
            "hyperparameters": hyperparams,
            "hardware": hardware_string(),
            "notes": notes,
        }
        with open(os.path.join(RESULTS_DIR, f"{self.agent}_meta.json"), "w") as fh:
            json.dump(meta, fh, indent=2, default=str)
        self.save()
        return meta

## Loading saved weights
The TA only receives the `.ipynb` and HTML, not our `models/` folder. When a notebook runs with `TRAIN = False` and the weights file is missing, `fetch_weights` downloads it from our GitHub repo first.

In [ ]:
def fetch_weights(agent_name, ext="pt", base_url=WEIGHTS_BASE_URL):
    """
    Return the local path to models/<agent_name>.<ext>, downloading it from
    base_url if it isn't on disk yet. Load it with your framework afterwards, e.g.
        net.load_state_dict(torch.load(fetch_weights("pong_full"), map_location="cpu"))
    """
    path = os.path.join(MODELS_DIR, f"{agent_name}.{ext}")
    if not os.path.exists(path):
        url = f"{base_url}/{agent_name}.{ext}"
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, path)
    return path

## Evaluation
`evaluate` produces the numbers we report and the TA re-runs. Game *i* is always reset with seed `EVAL_BASE_SEED + i`, so the same 200 games are played every time. Score = sum of rewards in a game: for Pong that's (our points − opponent points), so a mean above 0 is a winning score.

In [ ]:
def evaluate(env_fn, policy, n_games=EVAL_GAMES, agent_name=None,
             base_seed=EVAL_BASE_SEED, max_steps=50_000, verbose=True):
    """
    Play n_games full games with a trained policy (no learning, no exploration)
    and return a NumPy array of per-game scores.

    env_fn     : function returning a fresh env, e.g. lambda: make_env("Pong")
    policy     : object with .act(obs) -> action and optional .reset()
                 (a plain function obs -> action also works). Must be greedy.
    agent_name : if given, writes results/<agent_name>_eval_<n_games>.csv
    base_seed  : game i is reset with seed base_seed + i
    max_steps  : safety cap on agent steps per game, so a stalled game still ends
    """
    act = policy.act if hasattr(policy, "act") else policy
    env = env_fn()
    scores, lengths = [], []
    for i in range(n_games):
        obs, info = env.reset(seed=base_seed + i)
        env.action_space.seed(base_seed + i)
        if hasattr(policy, "reset"):
            policy.reset()
        total, steps, done = 0.0, 0, False
        while not done and steps < max_steps:
            obs, r, terminated, truncated, info = env.step(act(obs))
            total += r
            steps += 1
            done = terminated or truncated
        scores.append(total)
        lengths.append(steps)
        if verbose and (i + 1) % 20 == 0:
            print(f"  eval {i + 1}/{n_games}: running mean = {np.mean(scores):.2f}")
    env.close()
    scores = np.array(scores)
    if agent_name:
        pd.DataFrame({"game": np.arange(1, n_games + 1), "score": scores,
                      "steps": lengths}).to_csv(
            os.path.join(RESULTS_DIR, f"{agent_name}_eval_{n_games}.csv"), index=False)
    if verbose:
        print(f"{agent_name or 'policy'}: mean {scores.mean():.2f} +/- {scores.std():.2f} "
              f"over {n_games} games (games with score > 0: {(scores > 0).sum()})")
    return scores


class RandomPolicy:
    """Uniform random actions: the 'no skill' baseline. Seeded, so it is reproducible."""
    def __init__(self, n_actions, seed=0):
        self.n = n_actions
        self.rng = np.random.default_rng(seed)

    def act(self, obs):
        return int(self.rng.integers(self.n))


def random_baseline(game, n_games=EVAL_GAMES):
    """Evaluate a random agent and save results/<game>_random_eval_<n_games>.csv."""
    n_actions = make_env(game).action_space.n
    return evaluate(lambda: make_env(game), RandomPolicy(n_actions),
                    n_games=n_games, agent_name=f"{game.lower()}_random")

## Smoke test
Set `RUN_SMOKE_TEST = True` once to check that your install works. It plays 3 random Pong games twice and confirms both runs give the same scores (it writes a throwaway `results/pong_random_eval_3.csv`).

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    set_global_seeds(0)
    print("device:", get_device(), "|", hardware_string())
    for g in ["Pong", "Breakout"]:
        env = make_env(g, seed=0)
        obs, _ = env.reset(seed=0)
        print(g, "obs", obs.shape, "actions", env.unwrapped.get_action_meanings())
    pong_obs, _ = make_env("Pong").reset(seed=0)
    print("pong processed:", preprocess_pong(pong_obs).shape,
          "| pong stacked:", FrameStack(preprocess_pong).reset(pong_obs).shape,
          "| breakout processed:", preprocess_breakout(obs).shape)
    a = random_baseline("Pong", n_games=3)
    b = random_baseline("Pong", n_games=3)
    print("reproducible:", np.array_equal(a, b))

---
# Your agent starts here
Everything above this line is shared. Leave it unchanged in your copy. Put your agent's code below.